In [0]:
from pyspark.sql import functions as F

GOLD_CURRENT_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "gold/current_wildfire_risk"
)

GOLD_HISTORY_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "gold/wildfire_risk_history"
)

current = (
    spark.read
    .format("delta")
    .load(GOLD_CURRENT_PATH)
)

history = (
    spark.read
    .format("delta")
    .load(GOLD_HISTORY_PATH)
)

# -----------------------------------------
# Core checks
# -----------------------------------------

row_count = current.count()

unique_zones = (
    current
    .select("zone_id")
    .distinct()
    .count()
)

duplicate_zones = (
    current
    .groupBy("zone_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_weather = (
    current
    .filter(
        F.col("temperature_f").isNull()
        |
        F.col("relative_humidity_pct").isNull()
        |
        F.col("wind_speed_mph").isNull()
        |
        F.col("wind_gust_mph").isNull()
    )
    .count()
)

invalid_humidity = (
    current
    .filter(
        (F.col("relative_humidity_pct") < 0)
        |
        (F.col("relative_humidity_pct") > 100)
    )
    .count()
)

negative_wind = (
    current
    .filter(
        (F.col("wind_speed_mph") < 0)
        |
        (F.col("wind_gust_mph") < 0)
    )
    .count()
)

missing_risk = (
    current
    .filter(
        F.col("risk_score").isNull()
    )
    .count()
)

invalid_risk = (
    current
    .filter(
        (F.col("risk_score") < 0)
        |
        (F.col("risk_score") > 100)
    )
    .count()
)

valid_categories = [
    "Low",
    "Moderate",
    "High",
    "Critical"
]

invalid_categories = (
    current
    .filter(
        ~F.col("risk_category").isin(
            valid_categories
        )
    )
    .count()
)

# -----------------------------------------
# Results
# -----------------------------------------

print("DATA QUALITY REPORT")
print("-------------------")

print("Rows:", row_count)
print("Unique zones:", unique_zones)
print("Duplicate zones:", duplicate_zones)

print("Missing weather:", missing_weather)
print("Invalid humidity:", invalid_humidity)
print("Negative wind values:", negative_wind)

print("Missing risk scores:", missing_risk)
print("Invalid risk scores:", invalid_risk)
print("Invalid categories:", invalid_categories)

print(
    "History rows:",
    history.count()
)

print(
    "History snapshots:",
    history
    .select("pipeline_run_timestamp")
    .distinct()
    .count()
)

In [0]:
GOLD_CURRENT_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "gold/current_wildfire_risk"
)

GOLD_HISTORY_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "gold/wildfire_risk_history"
)

current = (
    spark.read
    .format("delta")
    .load(GOLD_CURRENT_PATH)
)

history = (
    spark.read
    .format("delta")
    .load(GOLD_HISTORY_PATH)
)

In [0]:
current.select(
    "pipeline_run_timestamp"
).distinct().show(truncate=False)

history.select(
    "pipeline_run_timestamp"
).distinct().orderBy(
    "pipeline_run_timestamp"
).show(truncate=False)